# Data Preparation

With a better understanding of data representation, let's now turn to preparing data for input into a machine learning pipeline. In the case of unsupervised learning, a simple matrix-level representation can suffice for input to machine learning models; we also need accompanying labels.

Often, traffic capture datasets are accompanied by labels. These labels can tell us something about the accompanying data points (i.e., flows, packets) in the traffic, and can be used to train the model for future prediction.

Automated tools exist for assigning labels to traffic flows, including [pcapML](https://nprint.github.io/pcapml/). Before we use those tools, we will do some automatic preparation and labeling from an existing dataset, a log4j trace from [malware traffic analysis](https://www.malware-traffic-analysis.net/2021/12/20/index.html) and a regular trace.

You can use the NetML traffic library to generate features.

In [2]:
import logging
logging.getLogger("scapy.runtime").setLevel(logging.ERROR)

from netml.pparser.parser import PCAP
from netml.utils.tool import dump_data, load_data

import pandas as pd

## Load the Packet Capture Files

Load the Log4j and HTTP packet capture files and extract features from the flows. You can feel free to compute features manually, although it will likely be more convenient at this point to use the `netML` library.

In [3]:
data_dir = "/home/frank/ml-systems/docs/notebooks/data"

# Load the two packet captures: log4j (malicious) and http (regular traffic)
log4j_pcap = PCAP(f"{data_dir}/log4j.pcap", flow_ptks_thres=2, random_state=42, verbose=0)
http_pcap = PCAP(f"{data_dir}/http.pcap", flow_ptks_thres=2, random_state=42, verbose=0)

## Convert the Packet Capture Into Flows

Find the function in `netml` that converts the pcap file into flows. Examing the resulting data structure. What does it contain?

In [4]:
# pcap2flows() parses the pcap and stores the result in .flows:
# a list of (flow_id, packets) tuples, where flow_id = (src_ip, dst_ip, src_port, dst_port, protocol)
log4j_pcap.pcap2flows()
http_pcap.pcap2flows()

flow_id, packets = log4j_pcap.flows[0]
print("log4j example flow:", flow_id, "with", len(packets), "packets")
flow_id, packets = http_pcap.flows[0]
print("http  example flow:", flow_id, "with", len(packets), "packets")

log4j example flow: ('128.14.134.170', '198.71.247.91', 57468, 80, 6) with 6 packets
http  example flow: ('192.168.1.58', '35.186.224.43', 57256, 443, 6) with 4 packets


## Explore the Flows

How many flows are in each of these pcaps? (Use the `netml` library output to determine the size of each data structure.)

In [5]:
print("Number of log4j flows:", len(log4j_pcap.flows))
print("Number of http flows: ", len(http_pcap.flows))

# Extract features from each flow: one row per flow, as a numpy array
log4j_pcap.flow2features("IAT", fft=False, header=False)
http_pcap.flow2features("IAT", fft=False, header=False)

log4j_features = pd.DataFrame(log4j_pcap.features)
http_features = pd.DataFrame(http_pcap.features)

print("log4j feature shape:", log4j_features.shape)
print("http feature shape: ", http_features.shape)
print(log4j_features.head())
print(http_features.head())

Number of log4j flows: 4795
Number of http flows:  291
log4j feature shape: (4795, 5)
http feature shape:  (291, 92)
          0         1         2         3         4
0  0.011543  0.000050  0.012709  5.041104  0.097399
1  0.011586  0.001091  5.004334  0.145770  0.000000
2  1.001059  2.020394  4.253039  0.000000  0.000000
3  1.816275  1.000564  1.000517  1.000615  1.000442
4  0.000078  0.000000  0.000000  0.000000  0.000000
          0          1          2         3    4    5    6    7    8    9   \
0   0.036961  30.002777   0.036621  0.000000  0.0  0.0  0.0  0.0  0.0  0.0   
1   0.019537  30.020075   0.019952  0.000000  0.0  0.0  0.0  0.0  0.0  0.0   
2  15.493915   4.998567  15.130779  5.336499  0.0  0.0  0.0  0.0  0.0  0.0   
3  15.433479   5.057394  15.072397  5.398758  0.0  0.0  0.0  0.0  0.0  0.0   
4   0.057702   0.000000   0.000000  0.000000  0.0  0.0  0.0  0.0  0.0  0.0   

   ...   82   83   84   85   86   87   88   89   90   91  
0  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  

## Normalize the Shapes of Each Feature Set

If you loaded the two pcaps with `netml` separately, the features will not be of the same dimension.  

1. Adjust your data frames so that the two have the same number of columns.
2. Merge (i.e., concatenate) the two data frames, but preserve the labels as a separate vector called "target".

In [6]:
# Pad the narrower feature set with zeros so both have the same number of columns
n_cols = max(log4j_features.shape[1], http_features.shape[1])
log4j_padded = log4j_features.reindex(columns=range(n_cols), fill_value=0)
http_padded = http_features.reindex(columns=range(n_cols), fill_value=0)

# Label column: 0 = http (regular), 1 = log4j (malicious)
http_padded["label"] = 0
log4j_padded["label"] = 1

# Merge into a single matrix: feature columns 0..n_cols-1, plus the label column
data = pd.concat([http_padded, log4j_padded], ignore_index=True)

# Also keep the labels as a separate vector
target = data["label"]

print("Merged shape:", data.shape)
print(data["label"].value_counts())
data.head()

Merged shape: (5086, 93)
label
1    4795
0     291
Name: count, dtype: int64


,0,1,2,3,4,5,6,7,8,9,...,83,84,85,86,87,88,89,90,91,label
0,0.036961,30.002777,0.036621,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.019537,30.020075,0.019952,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,15.493915,4.998567,15.130779,5.336499,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,15.433479,5.057394,15.072397,5.398758,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,0.057702,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


## Try Your Data on a Model

You should now have data that can be input into a model with scikit-learn. Import the scikit-learn package (`sklearn`) and a classification model of your choice to test that you can train your model with the above data. 

Hint: The function you want to call is `fit`.

**Note:** If you plan to use a linear model such as logistic regression, your label should be a numerical value, and if the problem is a binary classification model, as in this case, then the appropriate label should be 0 and 1 for each respective class. (If you are using a tree-based model, then the labels could take any format.)

(Note that we have not done anything here except train the model with all of the data. To evaluate the model, we will need to split the data into train and test sets.)

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix

X = data.drop(columns="label")
y = data["label"]

# Hold out 30% for testing. stratify=y keeps the http/log4j ratio the same in both splits,
# so both classes are present in training and testing.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
print("train:", y_train.value_counts().to_dict(), " test:", y_test.value_counts().to_dict())

# Logistic regression; features are standardized first because the IATs span several
# orders of magnitude (scaler is fit on the training data only).
# Try class_weight="balanced" to pay more attention to the rare http class.
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

print("train accuracy:", round(model.score(X_train, y_train), 4))
print("test accuracy: ", round(model.score(X_test, y_test), 4))

y_pred = model.predict(X_test)
print("\nConfusion matrix (rows = true, cols = predicted; order: http, log4j)")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["http", "log4j"], digits=3))

train: {1: 3356, 0: 204}  test: {1: 1439, 0: 87}
train accuracy: 0.9736
test accuracy:  0.9692

Confusion matrix (rows = true, cols = predicted; order: http, log4j)
[[  40   47]
 [   0 1439]]
              precision    recall  f1-score   support

        http      1.000     0.460     0.630        87
       log4j      0.968     1.000     0.984      1439

    accuracy                          0.969      1526
   macro avg      0.984     0.730     0.807      1526
weighted avg      0.970     0.969     0.964      1526



## Test Your Trained Model

We used the entire dataset to train the model in this example (no split), and so of course the model will be well-fit to all of the data. To simply test that your trained model works, call `predict` using a feature vector that you generate by hand (e.g., from scratch, using a random set of numbers, from another pcap).

In [8]:
import numpy as np

class_names = {0: "http", 1: "log4j"}

def predict_one(iats, description):
    """Predict the class of one flow given its inter-arrival times (zero-padded to 92 columns)."""
    vec = np.zeros(X.shape[1])
    vec[:len(iats)] = iats
    row = pd.DataFrame([vec], columns=X.columns)
    pred = model.predict(row)[0]
    p_log4j = model.predict_proba(row)[0, 1]
    print(f"{description:40s} -> {class_names[pred]}  (P(log4j) = {p_log4j:.3f})")

rng = np.random.default_rng(0)

# 1. Short flow with 5 inter-arrival times, similar to the log4j flows
predict_one([0.01, 0.0005, 0.01, 5.0, 0.1], "short flow, 5 IATs")

# 2. Longer flow with 40 random inter-arrival times
predict_one(rng.exponential(0.5, 40), "long flow, 40 random IATs")

# 3. Random values in every column
predict_one(rng.uniform(0, 1, 92), "random IATs in every column")

short flow, 5 IATs                       -> log4j  (P(log4j) = 0.984)
long flow, 40 random IATs                -> http  (P(log4j) = 0.000)
random IATs in every column              -> http  (P(log4j) = 0.000)


## Bonus  

Consider the following extensions to the above exercise:

* Concatenate or combine multiple features (either from `netml` or some of your own) into the same feature representation.
* Normalize your features so that they are in the same range (helpful for some models).

The above exercise gives you an example of how to generate features from a packet capture, attach labels to the dataset, and train a model using the labeled data. 

## Looking Ahead

Many other steps exist in the machine learning pipeline, including splitting the data into training and test sets, tuning model parameters, and evaluating the model. These will be the next steps we walk through.